# VAYU Climate Digital Twin â€” Kaggle GPU Training

**Accelerator**: GPU T4 Ã— 1 (16 GB) or P100 (16 GB)  
**Target**: Train VayuClimateModel on IMD 2010-2024, validate 2021-2023, test 2024  
**Dataset**: Upload `data/processed/` directory as Kaggle Dataset named `vayu-imd-processed`

## Setup
1. Enable GPU: Settings â†’ Accelerator â†’ GPU T4 x1
2. Add Dataset: `vayu-imd-processed` (your uploaded processed NetCDF files)
3. Run all cells top to bottom

In [ ]:
# â”€â”€ Environment check â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import subprocess, sys, os

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected â€” switch accelerator to GPU in Settings!')
print('Python:', sys.version)

In [ ]:
# â”€â”€ Install dependencies â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# torch-geometric version must match torch; use 2.5.3 for torch 2.x on Kaggle.
!pip install -q torch-geometric==2.5.3 xarray netcdf4 typer scipy
print('Dependencies installed')

In [ ]:
# ── Mount project code and locate dataset ─────────────────────────────────────
import sys, os
from pathlib import Path

REPO_DIR = '/kaggle/working/isro'
CHECKPOINT_DIR = f'{REPO_DIR}/checkpoints/wg_main'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Clone or pull — check for .git to detect a broken/partial directory
if os.path.exists(f'{REPO_DIR}/.git'):
    os.system(f'git -C {REPO_DIR} pull')
else:
    os.system(f'rm -rf {REPO_DIR}')   # remove partial/broken dir if any
    os.system(f'git clone https://github.com/Shyamistic/vayu.git {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working dir:', os.getcwd())

# Locate the uploaded Kaggle dataset (vayu-western-ghats-processed-v1)
root = Path('/kaggle/input')
required = [
    'rainfall_2010-2025.nc', 'tmax_2010-2025.nc', 'tmin_2010-2025.nc',
    'normalized_2010-2025.nc', 'pipeline_log_2010-2025.json',
]
found = {name: next(iter(root.rglob(name)), None) for name in required}
missing = [k for k, v in found.items() if v is None]
if missing:
    raise RuntimeError("Missing dataset files: " + ", ".join(missing) +
                       ". Attach the dataset via 'Add Input'.")
parent_counts = {}
for p in found.values():
    parent_counts[str(p.parent)] = parent_counts.get(str(p.parent), 0) + 1
DATASET_DIR = max(parent_counts, key=parent_counts.get)
print('Dataset dir:', DATASET_DIR)


In [ ]:
# ── Copy raw files + ancillary data (NCEP/CHIRPS/GEBCO) + build sequences ──────
import os, subprocess, sys, shutil
from pathlib import Path as _P
import xarray as xr, numpy as np

PY = sys.executable
os.makedirs(f'{REPO_DIR}/data/imd', exist_ok=True)
os.makedirs(f'{REPO_DIR}/data/processed_western_ghats', exist_ok=True)
os.makedirs(f'{REPO_DIR}/data/static', exist_ok=True)
os.makedirs(f'{REPO_DIR}/data/ncep_wind_subset', exist_ok=True)
os.makedirs(f'{REPO_DIR}/data/chirps_subset', exist_ok=True)

for f in ['rainfall_2010-2025.nc','tmax_2010-2025.nc','tmin_2010-2025.nc']:
    os.system(f'cp "{DATASET_DIR}/{f}" {REPO_DIR}/data/imd/')
for f in ['normalized_2010-2025.nc','pipeline_log_2010-2025.json']:
    os.system(f'cp "{DATASET_DIR}/{f}" {REPO_DIR}/data/processed_western_ghats/')

# ── Debug: show what Kaggle input datasets are mounted ────────────────────────
print('=== /kaggle/input contents ===')
for _d in sorted(_P('/kaggle/input').iterdir()):
    _nc_count = len(list(_d.rglob('*.nc')))
    print(f'  {_d.name}/  ({_nc_count} .nc files)')

# ── Ancillary: NCEP wind + CHIRPS (from vayu-ancillary-wg-v1) ───────────────
ncep_probe   = next(iter(_P('/kaggle/input').rglob('uwnd_2010_850hPa_WG.nc')), None)
chirps_probe = next(iter(_P('/kaggle/input').rglob('chirps_2010_WG.nc')), None)
gebco_hits   = (list(_P('/kaggle/input').rglob('GEBCO_*.nc'))
                + list(_P('/kaggle/input').rglob('gebco_*.nc')))
print(f'NCEP probe  : {ncep_probe}')
print(f'CHIRPS probe: {chirps_probe}')
print(f'GEBCO hits  : {gebco_hits}')

NCEP_DIR   = None
CHIRPS_DIR = None

if ncep_probe:
    NCEP_DIR = f'{REPO_DIR}/data/ncep_wind_subset'
    src = ncep_probe.parent
    for pattern in ('uwnd_*.nc', 'vwnd_*.nc', 'shum_*.nc', 'pr_wtr_*.nc'):
        for nc in src.glob(pattern):
            shutil.copy(nc, NCEP_DIR)
    print(f'NCEP: copied {len(list(_P(NCEP_DIR).glob("*.nc")))} files from {src}')
else:
    print('WARNING: No NCEP data — training without wind features (attach vayu-ancillary-wg-v1)')

if chirps_probe:
    CHIRPS_DIR = f'{REPO_DIR}/data/chirps_subset'
    src = chirps_probe.parent
    for nc in src.glob('chirps_*.nc'):
        shutil.copy(nc, CHIRPS_DIR)
    print(f'CHIRPS: copied {len(list(_P(CHIRPS_DIR).glob("*.nc")))} files')
else:
    print('WARNING: No CHIRPS data — training without CHIRPS blend')

# ── Preprocess: regenerate normalized data with ancillary features ────────────
preprocess_cmd = [
    PY, '-m', 'data_ingestion.cli', 'preprocess',
    '--data-dir',   f'{REPO_DIR}/data/imd',
    '--output-dir', f'{REPO_DIR}/data/processed_western_ghats',
    '--start-year', '2010', '--end-year', '2025',
    '--region', 'western_ghats', '--resolution', '0.25',
]
if NCEP_DIR:
    preprocess_cmd += ['--ncep-wind-dir', NCEP_DIR]
if CHIRPS_DIR:
    preprocess_cmd += ['--chirps-dir', CHIRPS_DIR]
subprocess.run(preprocess_cmd, check=True, cwd=REPO_DIR)

# ── GEBCO elevation (processed AFTER preprocess so grid is guaranteed to match) ──
ELEV_FILE = f'{REPO_DIR}/data/static/elevation_0.25deg.nc'
LSM_FILE  = f'{REPO_DIR}/data/static/lsm_0.25deg.nc'

if gebco_hits and not _P(ELEV_FILE).exists():
    # Load freshly-generated normalized file to get exact target grid
    _norm = xr.open_dataset(f'{REPO_DIR}/data/processed_western_ghats/normalized_2010-2025.nc')
    _tgt_lats = _norm.lat.values
    _tgt_lons = _norm.lon.values
    _norm.close()
    gebco = xr.open_dataset(str(gebco_hits[0]))
    elev_raw = gebco['elevation'].astype('float32')
    elev_025 = elev_raw.interp(lat=_tgt_lats, lon=_tgt_lons, method='linear').fillna(0.0)
    xr.Dataset({'elevation': elev_025}).to_netcdf(ELEV_FILE)
    lsm_025 = xr.where(elev_025 >= 0, 1.0, 0.0).astype('float32').rename('lsm')
    xr.Dataset({'lsm': lsm_025}).to_netcdf(LSM_FILE)
    print(f'GEBCO → elev {float(elev_025.min()):.0f}–{float(elev_025.max()):.0f} m on {len(_tgt_lats)}×{len(_tgt_lons)} grid')
elif gebco_hits:
    print('GEBCO already processed')
else:
    print('WARNING: No GEBCO file — using synthetic elevation')
    ELEV_FILE = None
    LSM_FILE  = None

# ── Pre-check: show normalized dataset shape/vars ──────────────────────────────
_nc = xr.open_dataset(f'{REPO_DIR}/data/processed_western_ghats/normalized_2010-2025.nc')
print('Normalized vars:', list(_nc.data_vars))
print('Grid: lat', len(_nc.lat), '| lon', len(_nc.lon), '| time', len(_nc.time))
_nc.close()

# ── Build 1024/256 sequences ─────────────────────────────────────────────────────
seq_cmd = [PY, '-m', 'data_ingestion.cli', 'build-sequences',
    '--normalized-file', f'{REPO_DIR}/data/processed_western_ghats/normalized_2010-2025.nc',
    '--output-dir',      f'{REPO_DIR}/data/processed_western_ghats',
    '--input-window', '30', '--target-window', '7',
    '--max-train', '1024', '--max-val', '256',
    '--stride', '2', '--fillna-value', '0.0']
if ELEV_FILE:
    seq_cmd += ['--elevation-file', ELEV_FILE, '--lsm-file', LSM_FILE]
# Capture both stdout and stderr so errors are visible
_res = subprocess.run(seq_cmd, cwd=REPO_DIR, capture_output=True, text=True)
print(_res.stdout[-3000:] if _res.stdout else '')
if _res.returncode != 0:
    print('\n=== build-sequences STDERR ===')
    print(_res.stderr[-4000:] if _res.stderr else '(empty)')
    raise RuntimeError(f'build-sequences failed (exit {_res.returncode})')

os.system(f'ls -lah {REPO_DIR}/data/processed_western_ghats')


In [ ]:
# -- Smoke check: verify model and data before full training ----------------
import subprocess, sys
PY = sys.executable

subprocess.run([PY,'-m','ai_engine.trainer',
    '--data-dir', f'{REPO_DIR}/data/processed_western_ghats',
    '--checkpoint-dir', f'{REPO_DIR}/checkpoints/wg_smoke',
    '--epochs','1','--device','auto','--smoke-only'],
    check=True, cwd=REPO_DIR)


In [ ]:
# ── FINAL best-quality training run (before Aurora) ───────────────────────────
# Target: R²_tmax ≥ 0.90, R²_rain as high as possible, no overfitting
#
# Key settings vs previous run:
#   --cosine-lr              : smooth LR decay → better convergence (replaces ReduceLROnPlateau)
#   --early-stopping-patience 15 : more patience to find real optimum
#   --weight-decay 1e-4      : stronger regularisation (was 1e-5)
#   --gnn-dropout 0.15       : slight regularisation boost (was 0.1)
#   --lambda-conservation 0.3: strong water-balance enforcement
#   --lambda-smoothness 0.01 : allows sharp orographic gradients at Ghats ridge
#   loss_functions.py        : rainfall weight=1.5, log1p transform
#   sequences                : real GEBCO elevation if attached (baked into graphs)
import os, subprocess, sys
PY = sys.executable
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

subprocess.run([PY,'-m','ai_engine.trainer',
    '--data-dir',               f'{REPO_DIR}/data/processed_western_ghats',
    '--checkpoint-dir',         CHECKPOINT_DIR,
    '--epochs',                 '80',
    '--device',                 'auto',
    '--amp',
    '--batch-size',             '1',
    '--grad-accum-steps',       '8',
    '--cosine-lr',
    '--early-stopping-patience','15',
    '--weight-decay',           '1e-4',
    '--gnn-dropout',            '0.15',
    '--lambda-conservation',    '0.3',
    '--lambda-smoothness',      '0.01',
    '--norm-params-file',       f'{REPO_DIR}/data/processed_western_ghats/norm_params_2010-2025.nc',
    '--run-baselines',
    '--require-benchmarks'],
    check=True, cwd=REPO_DIR)

os.system(f'ls -lah {CHECKPOINT_DIR}')


In [ ]:
# ── Load history and plot training curves ─────────────────────────────────────
import json, matplotlib.pyplot as plt
from pathlib import Path

history_path = Path(CHECKPOINT_DIR) / 'training_history.json'
if not history_path.exists():
    print('No training_history.json yet — run the training cell first.')
else:
    history = json.loads(history_path.read_text())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history['epochs'], history['train_loss'], label='Train Loss')
    axes[0].plot(history['epochs'], history['val_loss'],   label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('VAYU Training Loss')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history['epochs'], history['val_r2'], color='green', label='R² Tmax')
    axes[1].axhline(0.85, color='red', linestyle='--', label='Target R²=0.85')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('R²')
    axes[1].set_title('Validation R²')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves.png', dpi=150)
    plt.show()
    print('Best val_loss:', min(history['val_loss']))


In [ ]:
# â”€â”€ Save best checkpoint for download â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import shutil
from pathlib import Path

best_ckpt = Path(CHECKPOINT_DIR) / 'vayu_best.pt'
if best_ckpt.exists():
    shutil.copy(best_ckpt, '/kaggle/working/vayu_best.pt')
    size_mb = best_ckpt.stat().st_size / 1e6
    print(f'Checkpoint ready: /kaggle/working/vayu_best.pt ({size_mb:.1f} MB)')
    print('Download and upload to S3:')
    print('  aws s3 cp vayu_best.pt s3://vayu-models/checkpoints/')
else:
    print('vayu_best.pt not found â€” check training cell output for errors.')

## Next Steps After Training

1. **Download** `vayu_best.pt` from Kaggle Output
2. **Upload to S3**:
   ```bash
   aws s3 cp vayu_best.pt s3://vayu-climate-models/checkpoints/vayu_best.pt
   ```
3. **Trigger ECS deployment** (CDK will mount S3 checkpoint automatically)
4. **Verify**: `curl https://api.vayu-climate.com/health`

## Kaggle Quota Tips
- Each run â‰ˆ 2-4 hours on T4 (30h/week quota)
- Save checkpoint every 5 epochs to resume if quota runs out
- Use `early_stopping_patience=15` to auto-stop when converged
- Enable Accelerator **T4 x2** for 2Ã— speed if available